<a href="https://colab.research.google.com/github/regitaaft/iris-eda-ml/blob/main/ML_tugas_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loan Approval
Loan approval adalah data tentang pengajuan pinjaman (kredit) seseorang ke bank/lembaga keuangan dan apakah pengajuan itu disetujui atau ditolak.

Dataset yang digunakan telah memiliki label berupa status persetujuan pinjaman. Oleh karena itu, proses klasifikasi dilakukan untuk membangun model yang mampu mempelajari pola dari data tersebut dan meniru keputusan yang telah ada.

Akurasi digunakan sebagai metrik evaluasi untuk mengukur sejauh mana hasil prediksi model sesuai dengan data aktual. Dengan demikian, semakin tinggi nilai akurasi, maka semakin baik kemampuan model dalam memprediksi status persetujuan pinjaman.

### import library

In [ ]:
# Library dasar
import pandas as pd
import numpy as np

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & split data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Boosting
from xgboost import XGBClassifier

pandas → baca & olah data

numpy → operasi numerik

sklearn → machine learning

RandomForest → bagging

XGBoost → boosting

MLPClassifier → neural network

### Load Dataset

In [35]:
df = pd.read_csv('loan_data_set.csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


Dataset yang digunakan dalam penelitian ini adalah Loan Approval Dataset yang berisi data karakteristik pemohon pinjaman.

Berdasarkan output di atas, dapat dilihat bahwa dataset terdiri dari beberapa variabel, antara lain:

1.  ApplicantIncome: pendapatan pemohon
2.   LoanAmount: jumlah pinjaman yang diajukan
3. Credit_History: riwayat kredit pemohon
4. Dependents: jumlah tanggungan yang dimiliki pemohon (misalnya anak atau keluarga yang dibiayai)
5. Education: tingkat pendidikan pemohon (Graduate = lulusan sarjana, Not Graduate = bukan sarjana)
6. Self_Employed: status pekerjaan pemohon (Yes = wiraswasta, No = bukan wiraswasta)
7. Loan_Status: status persetujuan pinjaman (target), dengan kategori Y (disetujui) dan N (ditolak)



### Cek Data

In [45]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    int64  
 1   Gender             614 non-null    int64  
 2   Married            614 non-null    int64  
 3   Dependents         614 non-null    int64  
 4   Education          614 non-null    int64  
 5   Self_Employed      614 non-null    int64  
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    int64  
 12  Loan_Status        614 non-null    int64  
dtypes: float64(4), int64(9)
memory usage: 62.5 KB


,0
Loan_ID,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14


Masih terdapat missing value

LoanAmount	22

Loan_Amount_Term	14

Credit_History	50

### Data Cleaning

In [57]:
df.fillna(df.mode().iloc[0], inplace=True)
print(df.isnull().sum())

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


### Encoding

In [58]:
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])
print(df.head())
print(df.dtypes)

   Loan_ID  Gender  Married  Dependents  Education  Self_Employed  \
0        0       1        0           0          0              0   
1        1       1        1           1          0              0   
2        2       1        1           0          0              1   
3        3       1        1           0          1              0   
4        4       1        0           0          0              0   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0       120.0             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History  Property_Area  Loan_Status  
0             1.0              2            1  
1             1.0              0            0  
2             

### Pisahkan Fitur & Target

In [59]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']
print("X:", X.shape)
print("y:", y.shape)

X: (614, 12)
y: (614,)


Berdasarkan hasil pemisahan data, diperoleh variabel fitur (X) dengan ukuran (614, 12) yang terdiri dari 614 data dan 12 variabel independen. Sementara itu, variabel target (y) memiliki ukuran (614,) yang merepresentasikan 614 data status persetujuan pinjaman.

Variabel X digunakan sebagai input dalam model machine learning, sedangkan variabel y digunakan sebagai output yang akan diprediksi oleh model.


### Split Data

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (491, 12)
Test : (123, 12)


Total data: 614

80% → 491 (training)

20% → 123 (testing)

### Scaling (khusus NN)
Pada tahap preprocessing dilakukan proses standarisasi data menggunakan StandardScaler. Proses ini bertujuan untuk menyamakan skala antar variabel, karena pada dataset terdapat perbedaan rentang nilai yang cukup besar, seperti variabel *ApplicantIncome* yang bernilai ribuan, sedangkan variabel lain seperti *Credit_History* hanya bernilai 0 atau 1.

Standarisasi dilakukan dengan menghitung rata-rata dan standar deviasi dari data training, kemudian digunakan untuk mentransformasikan baik data training maupun data testing. Hal ini penting agar model machine learning, khususnya Neural Network, dapat bekerja secara optimal tanpa dipengaruhi oleh perbedaan skala antar variabel.


In [50]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## MODELING
### Random Forest (Bagging)

In [51]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.7723577235772358


Berdasarkan hasil pemodelan menggunakan algoritma Random Forest, diperoleh nilai akurasi sebesar 77,23%. Hal ini menunjukkan bahwa model mampu memprediksi status persetujuan pinjaman dengan tingkat ketepatan yang cukup baik.

Dengan demikian, Random Forest dapat digunakan sebagai salah satu model yang efektif dalam mengklasifikasikan kelayakan pemohon pinjaman berdasarkan karakteristik yang dimiliki.


### XGBoost (Boosting)

In [62]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

XGBoost Accuracy: 0.7642276422764228


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:28:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Berdasarkan hasil pemodelan menggunakan algoritma XGBoost, diperoleh nilai akurasi sebesar 76,42%. Hal ini menunjukkan bahwa model mampu mengklasifikasikan status persetujuan pinjaman dengan cukup baik.



### Neural Network

In [53]:
nn = MLPClassifier(hidden_layer_sizes=(10,10), max_iter=500)
nn.fit(X_train_scaled, y_train)

y_pred_nn = nn.predict(X_test_scaled)

print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))

Neural Network Accuracy: 0.7804878048780488


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


### Evaluasi Model

In [54]:
print("=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf))

print("=== XGBOOST ===")
print(classification_report(y_test, y_pred_xgb))

print("=== NEURAL NETWORK ===")
print(classification_report(y_test, y_pred_nn))

=== RANDOM FOREST ===
              precision    recall  f1-score   support

           0       0.86      0.42      0.56        43
           1       0.75      0.96      0.85        80

    accuracy                           0.77       123
   macro avg       0.81      0.69      0.70       123
weighted avg       0.79      0.77      0.75       123

=== XGBOOST ===
              precision    recall  f1-score   support

           0       0.77      0.47      0.58        43
           1       0.76      0.93      0.84        80

    accuracy                           0.76       123
   macro avg       0.77      0.70      0.71       123
weighted avg       0.77      0.76      0.75       123

=== NEURAL NETWORK ===
              precision    recall  f1-score   support

           0       0.83      0.47      0.60        43
           1       0.77      0.95      0.85        80

    accuracy                           0.78       123
   macro avg       0.80      0.71      0.72       123
weighted avg 

Berdasarkan hasil evaluasi menggunakan classification report, seluruh model menunjukkan performa yang lebih baik dalam memprediksi kelas disetujui (Loan_Status = 1) dibandingkan dengan kelas ditolak (Loan_Status = 0).

Hal ini terlihat dari nilai recall yang tinggi pada kelas 1, yaitu di atas 0,90 pada semua model, yang menunjukkan bahwa sebagian besar data yang benar-benar disetujui berhasil diprediksi dengan baik. Namun, nilai recall pada kelas 0 relatif rendah, berkisar antara 0,42 hingga 0,47, yang menunjukkan bahwa model masih kesulitan dalam mengidentifikasi pemohon yang seharusnya ditolak.

Kondisi ini kemungkinan disebabkan oleh ketidakseimbangan jumlah data antara kelas disetujui dan ditolak, sehingga model cenderung lebih fokus pada kelas mayoritas.

Dari ketiga model yang digunakan, Neural Network menunjukkan performa terbaik dengan nilai akurasi sebesar 78,05% serta keseimbangan yang lebih baik antara precision dan recall dibandingkan model lainnya.


### Perbandingan Model

In [56]:
print("RF:", accuracy_score(y_test, y_pred_rf))
print("XGB:", accuracy_score(y_test, y_pred_xgb))
print("NN:", accuracy_score(y_test, y_pred_nn))

RF: 0.7723577235772358
XGB: 0.7642276422764228
NN: 0.7804878048780488


## Kesimpulan
Berdasarkan hasil perbandingan tiga algoritma machine learning, yaitu Random Forest, XGBoost, dan Neural Network, diperoleh bahwa Neural Network memiliki performa terbaik dengan nilai akurasi sebesar 78,05%.

Sementara itu, Random Forest menghasilkan akurasi sebesar 77,23% dan XGBoost sebesar 76,42%. Perbedaan nilai akurasi antar model tidak terlalu signifikan, yang menunjukkan bahwa ketiga model memiliki kemampuan yang cukup baik dalam memprediksi status persetujuan pinjaman.

Namun demikian, Neural Network dapat dipilih sebagai model terbaik karena memiliki tingkat akurasi paling tinggi dibandingkan model lainnya.
